# 02 - Build

NB03 (performans) ve NB04 (kontrol mekanizmasi) ayni turetmeyi iki kere
yapmasin diye episode/regime segmentasyonu, state, action sinifi, T0 ve
event tespiti burada bir kez uretilir.

**Iki ayri birim var, karistirilmamali:**

| Birim | Tanim | Ne icin |
|---|---|---|
| Episode | reset'ten reset'e | T0, sure (Ludolph) |
| Regime run | kuadran dizisi, theta*omega isaret degistirince yeni run | Safe/Saved/Failed (Park) |

Girdi: NB01'in yazdigi `samples_clean.parquet` ve `trials_clean.parquet`.

In [1]:
%pip install -q pyyaml pandas numpy pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

ANALYSIS_ROOT = Path.cwd()
if not (ANALYSIS_ROOT / "config.yaml").exists():
    ANALYSIS_ROOT = ANALYSIS_ROOT.parent
sys.path.insert(0, str(ANALYSIS_ROOT))

from src.physics import params_from_config, compute_T0, verify_model
from src.build import build_all, validate_T0_freefall

with open(ANALYSIS_ROOT / "config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)

INTERIM_DIR = ANALYSIS_ROOT / config["paths"]["interim_dir"]
pd.set_option("display.max_columns", None)

df_samples = pd.read_parquet(INTERIM_DIR / "samples_clean.parquet")
df_trials = pd.read_parquet(INTERIM_DIR / "trials_clean.parquet")
print(f"Sample: {len(df_samples):,}   Trial: {len(df_trials)}")

Sample: 499,560   Trial: 371


## 1. Fizik modeli dogrulamasi

T0 hesabi bir modele dayaniyor. Model varsayilmiyor: kayitli
`applied_force_n` ile simule edilen acisal ivme, gozlenen ivmeyle
karsilastiriliyor. Korelasyon dusukse T0'a guvenilmez.

In [3]:
p = params_from_config(config)
print("Parametreler:")
for k, v in p.items():
    print(f"  {k:18} {v}")

ver = verify_model(df_samples, p)
if ver.empty:
    print("\nDogrulanamadi.")
else:
    lo, hi = ver.corr_alpha.min(), ver.corr_alpha.max()
    print(f"\nModel dogrulama: {len(ver)} parca, korelasyon {lo:.4f} - {hi:.4f}")
    if lo < 0.95:
        print("UYARI: korelasyon dusuk, T0 guvenilir degil.")
    display(ver)

print("\nT0(theta0):")
for a in [0.5, 1, 2, 3, 5, 7, 7.5]:
    print(f"  {a:4.1f} deg -> {compute_T0(a, p):.2f} s")

Parametreler:
  m_c                0.4
  m_p                0.08
  l                  0.5
  g                  1.0
  k                  1.3333333333333333
  angle_limit_deg    60.0
  track_limit_m      5.0
  dt                 0.016666666666666666
  max_time_s         60.0
  round_decimals     4

Model dogrulama: 8 parca, korelasyon 0.9884 - 0.9972


,participant_id,trial_id,n,corr_alpha,rms_alpha
0,P001,T002,360,0.9947,0.2262
1,P001,T003,393,0.9963,0.1108
2,P001,T004,709,0.9959,0.2181
3,P001,T005,441,0.9966,0.1778
4,P001,T006,389,0.9967,0.2243
5,P001,T007,740,0.9952,0.1376
6,P001,T008,389,0.9972,0.1438
7,P001,T009,726,0.9884,0.2735



T0(theta0):
   0.5 deg -> 4.23 s
   1.0 deg -> 3.70 s
   2.0 deg -> 3.18 s
   3.0 deg -> 2.87 s
   5.0 deg -> 2.48 s
   7.0 deg -> 2.22 s
   7.5 deg -> 2.17 s


## 2. Turetme

`build_all` sirayla: state (kuadran) -> action sinifi -> episode ->
regime run -> event.

In [4]:
df_built, episodes, regimes, events = build_all(df_samples, df_trials, config)

print(f"sample  {df_built.shape}")
print(f"episode {episodes.shape}")
print(f"regime  {regimes.shape}")
print(f"event   {events.shape}")

sample  (499560, 31)
episode (1277, 25)
regime  (10855, 20)
event   (20428, 12)


## 2b. T0'in ampirik dogrulamasi

Katilimcinin hic girdi vermedigi ve aci limitiyle biten episode'lar tanim
geregi serbest dususturler. Bu episodelarda gozlenen sure T0'a **esit**
olmali. Fizik modeli, RK4 adimi, T0 hesabi ve episode segmentasyonu
zincirinin tamamini tek seferde test eder.

In [5]:
free = validate_T0_freefall(df_built, episodes)

if free.empty:
    print("Girdisiz, aci limitiyle biten episode bulunamadi - test yapilamadi.")
else:
    n = free.attrs["n"]
    dev = free.attrs["max_dev"]
    print(f"Serbest dusus episode'u: {n}")
    print(f"duration/T0 ortalama: {free.attrs['mean_ratio']:.4f}")
    print(f"1.0'dan max sapma:    {dev:.4f}")
    print("SONUC:", "T0 dogrulandi" if dev < 0.02 else "UYARI - T0 tutmuyor")
    display(free.round(4))

Serbest dusus episode'u: 5
duration/T0 ortalama: 1.0000
1.0'dan max sapma:    0.0000
SONUC: T0 dogrulandi


,participant_id,trial_id,episode,theta0_deg,omega0_deg_s,duration_s,T0_s,duration_over_T0
0,P001,T001,0.0,-6.4491,-0.1835,2.2834,2.2833,1.0
1,P001,T004,0.0,-2.1192,-0.0605,3.1334,3.1333,1.0
2,P004,T020,0.0,5.1891,0.1479,2.4500,2.4500,1.0
3,P007,T009,0.0,1.7142,0.0490,3.3001,3.3000,1.0
4,P007,T045,2.0,-3.7600,-0.1073,2.7001,2.7000,1.0


## 3. State ve action dagilimi

Isaret konvansiyonu veriden dogrulandi: duzeltici kuvvet theta ile
**ayni** isaretli. Park'in metnindekinin tersi, cunku onun VIP'inde
joystick dogrudan acisal ivme veriyor, bizde kuvvet cart'a gidiyor.

In [6]:
act = df_built[df_built["phase"] == "active"]

print("Kuadran (%):")
q = act["falling"].map({True: "fall", False: "safe"}).value_counts(normalize=True)
print((q * 100).round(1).to_string())

print()
print("Action sinifi (%):")
vc = (act["action"].value_counts(normalize=True) * 100).round(2)
display(vc.to_frame("pct"))

x = act[act["action"] == "X"]
if len(x):
    print(f"X ({len(x)} ornek, %{100 * len(x) / len(act):.2f}) sebepleri:")
    print(x["action_excluded_reason"].value_counts().to_string())

print()
print("Katilimci basina action (%):")
tab = act.groupby(["participant_id", "action"]).size().unstack(fill_value=0)
display((100 * tab.div(tab.sum(axis=1), axis=0)).round(1))

Kuadran (%):
falling
fall    64.2
safe    35.8

Action sinifi (%):


,pct
action,
I,73.51
CR,22.80
D,3.40
A,0.29
X,0.01


X (39 ornek, %0.01) sebepleri:
action_excluded_reason
transient_neutral    39

Katilimci basina action (%):


action,A,CR,D,I,X
participant_id,,,,,
P001,0.1,31.8,1.3,66.8,0.0
P002,0.5,12.1,0.2,87.2,0.0
P003,0.3,25.5,2.4,71.8,0.0
P004,0.1,23.9,3.3,72.7,0.0
P005,0.4,21.4,0.7,77.6,0.0
P006,0.4,24.6,0.7,74.3,0.0
P007,0.2,20.3,15.2,64.3,0.0


## 4. Episode

Reset'ten reset'e. `censored` = dususle degil trial bitisiyle sona erdi,
yani ne kadar daha dayanacagi bilinmiyor.

`fall_cause` iki degerli: **angle** (pole +-60 dereceye vardi) veya
**track** (cart +-5 m ray sinirina carpti). Ikisi ayri basarisizlik
turu; ray kaynakli dususlerde pole cogu zaman dik duruyor.

In [7]:
print(f"Episode: {len(episodes)}")
print(f"  dususle biten : {int(episodes.ended_in_fall.sum())}")
print(f"  sansurlu      : {int(episodes.censored.sum())}")
print()
print("Dusus sebebi:")
display(episodes.fall_cause.value_counts(dropna=False).to_frame("n"))

print("Sebep basina dusus anindaki max |theta|:")
fall_ep = episodes[episodes.ended_in_fall]
display(fall_ep.groupby("fall_cause").max_abs_theta_deg.describe().round(2))

print()
print("Episode sure ve T0:")
display(episodes[["duration_s", "theta0_deg", "T0_s",
                  "duration_over_T0"]].describe().round(3))

n_nan = int(episodes.T0_s.isna().sum())
if n_nan:
    print(f"UYARI: {n_nan} episode'da T0 hesaplanamadi (theta0 ~ 0).")

Episode: 1277
  dususle biten : 906
  sansurlu      : 371

Dusus sebebi:


,n
fall_cause,
angle,814
NaN,371
track,92


Sebep basina dusus anindaki max |theta|:


,count,mean,std,min,25%,50%,75%,max
fall_cause,,,,,,,,
angle,814.0,61.04,0.90,60.00,60.34,60.81,61.51,65.86
track,92.0,46.42,9.35,21.89,39.02,48.87,54.30,59.61



Episode sure ve T0:


,duration_s,theta0_deg,T0_s,duration_over_T0
count,1277.000,1277.000,1277.000,1277.000
mean,5.811,-0.342,2.928,2.051
std,5.919,4.312,0.793,2.150
min,0.033,-7.479,2.167,0.012
25%,1.833,-3.890,2.383,0.665
50%,3.300,-0.671,2.700,1.110
75%,7.450,3.564,3.200,2.603
max,20.000,7.427,9.683,9.160


## 5. Regime run

Park'in rejimleri. `Failed` sadece aci kaynakli dususler icin -- Park ile
karsilastirilabilir olan bu. `TrackLoss` ray kaybi, Park'ta karsiligi yok,
Park karsilastirmalarindan cikarilmali.

In [8]:
print(f"Regime run: {len(regimes)}")
n_ep = regimes.groupby(["participant_id", "trial_id", "episode"]).ngroups
print(f"Episode basina ortalama run: {len(regimes) / n_ep:.1f}")
print()
tab = regimes.regime.value_counts().to_frame("n")
tab["pct"] = (100 * tab.n / len(regimes)).round(1)
display(tab)

print("Rejim basina sure ve max |theta|:")
display(regimes.groupby("regime")[["duration_s", "max_abs_theta_deg"]].mean().round(2))

print()
print("Rejim basina action dagilimi (%):")
display(regimes.groupby("regime")[["pct_I", "pct_CR", "pct_A", "pct_D"]].mean().round(1))

min_n = config["build"]["regime_min_samples"]
short = regimes[regimes.n_samples < min_n]
print()
print(f"Cok kisa run (< {min_n} ornek): {len(short)}")

Regime run: 10855
Episode basina ortalama run: 8.5



,n,pct
regime,,
Safe,4873,44.9
Saved,4842,44.6
Failed,814,7.5
censored,234,2.2
TrackLoss,92,0.8


Rejim basina sure ve max |theta|:


,duration_s,max_abs_theta_deg
regime,,
Failed,1.05,61.04
Safe,0.54,17.15
Saved,0.77,17.41
TrackLoss,0.40,25.00
censored,0.65,15.72



Rejim basina action dagilimi (%):


,pct_I,pct_CR,pct_A,pct_D
regime,,,,
Failed,54.1,17.0,0.0,28.8
Safe,59.1,40.1,0.8,0.0
Saved,73.1,25.6,0.0,1.3
TrackLoss,53.9,32.7,0.4,12.7
censored,80.8,11.4,0.0,7.8



Cok kisa run (< 2 ornek): 80


## 6. Event

`onset` notr banddan cikis, `offset` banda donus, `reversal` kuvvet yon
degistirme (Ludolph'un action timing'i bunu kullanir), `fall` dusus.

Olaylar episode icinde araniyor. Reset satirlarinda `applied_force_n`
sifira zorlanip `input_applied` son degerinde kaldigi icin, parcalari uc
uca eklemek sahte zero-crossing uretirdi.

In [9]:
display(events.event.value_counts().to_frame("n"))

n_fall_ev = int((events.event == "fall").sum())
n_fall_ts = int(df_trials.fall_count.sum())
status = "TUTUYOR" if n_fall_ev == n_fall_ts else "TUTMUYOR"
print(f"fall event {n_fall_ev} vs trial_summary fall_count {n_fall_ts} -> {status}")

print()
print("Trial basina event (measurement, qc_pass):")
ok = df_trials[(df_trials.practice == 0) & df_trials.qc_pass]
ok = ok[["participant_id", "trial_id"]]
ev_ok = events.merge(ok, on=["participant_id", "trial_id"])
per = ev_ok.groupby(["participant_id", "trial_id"]).event.value_counts().unstack(fill_value=0)
display(per.mean().round(2).to_frame("trial basina ortalama"))

,n
event,
onset,8027
offset,7554
reversal,3941
fall,906


fall event 906 vs trial_summary fall_count 906 -> TUTUYOR

Trial basina event (measurement, qc_pass):


,trial basina ortalama
event,
fall,2.16
offset,20.35
onset,21.63
reversal,10.55


## 7. Ornek: bir episode'un run'lari

Segmentasyonun dogru calistigini gozle dogrulamak icin.

In [10]:
pid = episodes.participant_id.iloc[0]
sel = episodes[(episodes.participant_id == pid) & episodes.ended_in_fall]
if len(sel):
    tid = sel.trial_id.iloc[0]
    epi = sel.episode.iloc[0]
    print(f"{pid} / {tid} / episode {epi}")
    display(sel.iloc[[0]][["duration_s", "theta0_deg", "T0_s",
                           "duration_over_T0", "fall_cause"]].round(3))
    r = regimes[(regimes.participant_id == pid) & (regimes.trial_id == tid)
                & (regimes.episode == epi)]
    display(r[["run", "regime", "quadrant_type", "duration_s", "theta_start_deg",
               "theta_end_deg", "max_abs_theta_deg", "pct_I", "pct_CR",
               "pct_A", "pct_D"]])

P001 / T001 / episode 0


,duration_s,theta0_deg,T0_s,duration_over_T0,fall_cause
0,2.283,-6.449,2.283,1.0,angle


,run,regime,quadrant_type,duration_s,theta_start_deg,theta_end_deg,max_abs_theta_deg,pct_I,pct_CR,pct_A,pct_D
0,0,Failed,fall,2.2834,-6.4491,-60.5064,60.5064,100.0,0.0,0.0,0.0


## Cikti

In [11]:
df_built.drop(columns=["state_defined"], errors="ignore").to_parquet(
    INTERIM_DIR / "samples_built.parquet", index=False)
episodes.to_parquet(INTERIM_DIR / "episodes.parquet", index=False)
regimes.to_parquet(INTERIM_DIR / "regimes.parquet", index=False)
events.to_parquet(INTERIM_DIR / "events.parquet", index=False)

print(f"samples_built.parquet  ({len(df_built):,} satir)")
print(f"episodes.parquet       ({len(episodes):,} satir)")
print(f"regimes.parquet        ({len(regimes):,} satir)")
print(f"events.parquet         ({len(events):,} satir)")

samples_built.parquet  (499,560 satir)
episodes.parquet       (1,277 satir)
regimes.parquet        (10,855 satir)
events.parquet         (20,428 satir)
